In [2]:
!pip install transformers peft trl datasets accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.4 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
DATA_PATH = "/kaggle/input/datasets/ayushanand1009/ambiguity-dataset/hindi_ambiguity_dataset.csv"

df = pd.read_csv(DATA_PATH)
print(f"Total samples : {len(df)}")
print(f"Columns       : {df.columns.tolist()}")
print(f"\nCategory distribution:")
print(df["category"].value_counts())
df.head()

Total samples : 1000
Columns       : ['id', 'sentence', 'ambiguous_word', 'correct_meaning', 'category', 'language', 'source']

Category distribution:
category
time_ambiguity             133
object_action_ambiguity    112
hinglish_ambiguity         110
idiom_ambiguity            108
morphological_ambiguity    101
syntactic_ambiguity         96
pragmatic_ambiguity         90
pronoun_ambiguity           85
sarcasm_ambiguity           85
place_ambiguity             80
Name: count, dtype: int64


,id,sentence,ambiguous_word,correct_meaning,category,language,source
0,1,कल वह बाज़ार गया था।,कल,yesterday,time_ambiguity,Hindi,custom_annotated
1,2,कल मैंने एक अच्छी फिल्म देखी।,कल,yesterday,time_ambiguity,Hindi,custom_annotated
2,3,कल बहुत बारिश हुई थी।,कल,yesterday,time_ambiguity,Hindi,custom_annotated
3,4,कल उसने मुझे फोन किया था।,कल,yesterday,time_ambiguity,Hindi,custom_annotated
4,5,कल स्कूल में छुट्टी थी।,कल,yesterday,time_ambiguity,Hindi,custom_annotated


In [5]:
CATEGORY_DESC = {
    "time_ambiguity":           "time context (past/future)",
    "morphological_ambiguity":  "word form / part of speech",
    "syntactic_ambiguity":      "sentence structure",
    "pragmatic_ambiguity":      "context and intent",
    "pronoun_ambiguity":        "pronoun reference",
    "sarcasm_ambiguity":        "tone (literal vs sarcastic)",
    "idiom_ambiguity":          "idiomatic vs literal meaning",
    "hinglish_ambiguity":       "code-switched Hindi-English",
    "place_ambiguity":          "place name vs common noun",
    "object_action_ambiguity":  "object vs action",
}

def format_row(row):
    cat = CATEGORY_DESC.get(row["category"], row["category"])
    instruction = (
        "You are an NLP expert in Hindi language. "
        "Identify the correct meaning of the ambiguous word in the sentence."
    )
    user_input = (
        f"Sentence: {row['sentence']}\n"
        f"Ambiguous word: \"{row['ambiguous_word']}\"\n"
        f"Ambiguity type: {cat}"
    )
    response = (
        f"The word \"{row['ambiguous_word']}\" in this sentence means: "
        f"{row['correct_meaning'].replace('_', ' ')}.\n"
        f"Ambiguity type: {cat}."
    )
    return {
        "text": f"### Instruction:\n{instruction}\n\n### Input:\n{user_input}\n\n### Response:\n{response}"
    }

formatted = [format_row(row) for _, row in df.iterrows()]

print("Sample prompt (yesterday meaning):")
print(formatted[0]["text"])
print("\n" + "="*50 + "\n")
print("Sample prompt (tomorrow meaning):")
print(formatted[15]["text"])

Sample prompt (yesterday meaning):
### Instruction:
You are an NLP expert in Hindi language. Identify the correct meaning of the ambiguous word in the sentence.

### Input:
Sentence: कल वह बाज़ार गया था।
Ambiguous word: "कल"
Ambiguity type: time context (past/future)

### Response:
The word "कल" in this sentence means: yesterday.
Ambiguity type: time context (past/future).


Sample prompt (tomorrow meaning):
### Instruction:
You are an NLP expert in Hindi language. Identify the correct meaning of the ambiguous word in the sentence.

### Input:
Sentence: कल परीक्षा है, पढ़ाई करो।
Ambiguous word: "कल"
Ambiguity type: time context (past/future)

### Response:
The word "कल" in this sentence means: tomorrow.
Ambiguity type: time context (past/future).


In [6]:
hf_dataset = Dataset.from_list(formatted)
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
test_dataset  = split["test"]

print(f"Train samples : {len(train_dataset)}")
print(f"Test  samples : {len(test_dataset)}")

Train samples : 900
Test  samples : 100


In [7]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True,
)
model.config.use_cache = False

total = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded! Total parameters: {total:,} ({total/1e6:.1f}M)")

Loading tokenizer: Qwen/Qwen2.5-0.5B


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading model...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


Model loaded! Total parameters: 494,032,768 (494.0M)


In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"Total params     : {total:,}")
print("LoRA applied. Ready to train!")

Trainable params : 2,162,688 (0.44%)
Total params     : 496,195,456
LoRA applied. Ready to train!


In [10]:
sft_config = SFTConfig(
    output_dir="./hindi_ambiguity_qwen_lora",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    dataset_text_field="text",
    max_length=256,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting training...")
trainer.train()
print("Training complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Epoch,Training Loss,Validation Loss
1,1.097552,0.589975
2,0.491700,0.464893
3,0.420426,0.432593
4,0.380473,0.408807
5,0.354048,0.403336
6,0.331569,0.395764
7,0.315998,0.395539
8,0.301334,0.396671
9,0.291511,0.397560
10,0.293549,0.397738


Training complete!


In [11]:
OUTPUT_DIR = "./hindi_ambiguity_qwen_lora_final"
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")

Model saved to: ./hindi_ambiguity_qwen_lora_final


In [12]:
def predict_ambiguity(sentence, ambiguous_word, ambiguity_type="time_ambiguity"):
    cat = CATEGORY_DESC.get(ambiguity_type, ambiguity_type)
    prompt = (
        f"### Instruction:\n"
        f"You are an NLP expert in Hindi language. "
        f"Identify the correct meaning of the ambiguous word in the sentence.\n\n"
        f"### Input:\n"
        f"Sentence: {sentence}\n"
        f"Ambiguous word: \"{ambiguous_word}\"\n"
        f"Ambiguity type: {cat}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    full_out = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_out[len(prompt):].strip()

test_cases = [
    ("कल वह बाज़ार गया था।",         "कल",       "time_ambiguity"),
    ("कल हम दिल्ली जाएंगे।",         "कल",       "time_ambiguity"),
    ("उसके गले में सोने का हार था।", "सोने",     "morphological_ambiguity"),
    ("उसने कहा वह नहीं आएगा।",      "वह",       "pronoun_ambiguity"),
    ("हाँ, बहुत होशियार हो तुम!",    "होशियार",  "sarcasm_ambiguity"),
    ("वह पानी में मछली की तरह है।",  "मछली",     "idiom_ambiguity"),
    ("उसका mood बहुत off है आज।",    "mood",      "hinglish_ambiguity"),
]

print("=" * 60)
print("  HINDI AMBIGUITY RESOLUTION - MODEL OUTPUT")
print("=" * 60)
for sentence, word, atype in test_cases:
    print(f"\nSentence   : {sentence}")
    print(f"Word       : '{word}' ({atype})")
    print(f"Prediction : {predict_ambiguity(sentence, word, atype)}")
    print("-" * 60)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  HINDI AMBIGUITY RESOLUTION - MODEL OUTPUT

Sentence   : कल वह बाज़ार गया था।
Word       : 'कल' (time_ambiguity)
Prediction : The word "कल" in this sentence means: yesterday.
Ambiguity type: time context (past/future).
------------------------------------------------------------

Sentence   : कल हम दिल्ली जाएंगे।
Word       : 'कल' (time_ambiguity)
Prediction : The word "कल" in this sentence means: tomorrow.
Ambiguity type: time context (past/future).
------------------------------------------------------------

Sentence   : उसके गले में सोने का हार था।
Word       : 'सोने' (morphological_ambiguity)
Prediction : The word "सोने" in this sentence means: to sleep.
Ambiguity type: word form / part of speech.
------------------------------------------------------------

Sentence   : उसने कहा वह नहीं आएगा।
Word       : 'वह' (pronoun_ambiguity)
Prediction : The word "वह" in this sentence means: unclear friend.
Ambiguity type: pronoun reference.
-------------------------------------------------

In [14]:
correct = 0
total = 0

for sample in test_dataset.select(range(min(50, len(test_dataset)))):
    text = sample["text"]
    if "### Response:" not in text:
        continue

    gt_response = text.split("### Response:")[-1].strip().lower()
    input_part  = text.split("### Input:")[-1].split("### Response:")[0].strip()
    lines = input_part.split("\n")
    sentence = lines[0].replace("Sentence:", "").strip() if lines else ""
    word     = lines[1].replace("Ambiguous word:", "").strip().strip('"') if len(lines) > 1 else ""
    atype    = lines[2].replace("Ambiguity type:", "").strip() if len(lines) > 2 else ""

    pred = predict_ambiguity(sentence, word, atype).lower()

    # Ground truth se saare keywords nikalo
    gt_keywords = []
    if "means:" in gt_response:
        gt_meaning = gt_response.split("means:")[-1].split(".")[0].strip()
        gt_keywords = [w.strip() for w in gt_meaning.replace("_", " ").split()]

    # Agar koi bhi keyword match kare to correct
    matched = any(kw in pred for kw in gt_keywords if len(kw) > 2)
    if matched:
        correct += 1
    total += 1

print(f"Evaluation on {total} test samples")
print(f"Correct   : {correct}")
print(f"Incorrect : {total - correct}")
print(f"Accuracy  : {100*correct/total:.1f}%")

Evaluation on 50 test samples
Correct   : 28
Incorrect : 22
Accuracy  : 56.0%
